# Lab Day 1 — Mạng nơ-ron và thí nghiệm huấn luyện (Forest CoverType)

**Võ Minh Quân — MSSV 2A202602429**

Notebook chạy từ `submission_2A202602429/code/`: đọc dữ liệu ở `../../data`, ghi `../figures/`, `../results/`,
`../experiments.xlsx`, `../predictions_eval.csv`, `../eval_result.json`. Checkpoint (`.pt`) ghi ở `../../checkpoints/`
(ngoài thư mục nộp).

**Chạy trên Colab:** *Runtime → Change runtime type → T4 GPU*, rồi *Run all*. Ô đầu tiên mount Google Drive và clone repo vào Drive,
nên nếu Colab ngắt kết nối thì chạy lại *Run all*: các thí nghiệm đã xong được nạp lại từ `results/*.json` + checkpoint, không huấn luyện lại.

Quy tắc xuyên suốt: chọn mọi thứ (lr, cấu hình, epoch) **chỉ bằng val**; tập eval chỉ dùng ở Part 4 cho baseline và cấu hình cuối.

In [ ]:
# ===== Môi trường và đường dẫn =====
import os, sys, json, math, time, subprocess, shutil
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vminhquan/K4-Track4-Day1-VoMinhQuan-2A202602429.git"
SUBMISSION = "submission_2A202602429"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/AI_Track4_lab_day1"
    REPO_DIR = f"{BASE}/K4-Track4-Day1-VoMinhQuan-2A202602429"
    os.makedirs(BASE, exist_ok=True)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:  # lấy code mới nhất; results/, figures/, checkpoints/ không bị git đụng tới
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(f"{REPO_DIR}/{SUBMISSION}/code")

sys.path.insert(0, os.getcwd())
REPO_ROOT = "../.."   # chứa data/ và scripts/
OUT_DIR   = ".."      # = submission_2A202602429/
CKPT_DIR  = f"{REPO_ROOT}/checkpoints"
SMOKE = os.environ.get("LAB_SMOKE") == "1"   # chỉ dùng để kiểm thử code nhanh (dữ liệu nhỏ, 2 epoch); mặc định tắt
if SMOKE:
    OUT_DIR, CKPT_DIR = os.environ["LAB_OUT"], os.environ["LAB_OUT"] + "/ckpt"
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| SMOKE:", SMOKE)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, compute_loss
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
from runner import Runner

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("CẢNH BÁO: không có GPU CUDA. Thí nghiệm AMP sẽ bị bỏ qua; bật Runtime → Change runtime type → T4 GPU.")
for d in ("figures", "results"):
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
LN7 = math.log(7)

## Part 0 — Dữ liệu
1. `scripts/split_data.py` chia theo `split_metadata.csv` → `data/processed/train.npz` (464 809), `eval.npz` (116 203).
2. Tách val = 20% train, phân tầng, seed 42 → 371 847 train / 92 962 val. Cùng phép tách cho mọi thí nghiệm.
3. Chuẩn hoá 10 cột số bằng mean/std **của phần train còn lại**; 44 cột nhị phân giữ nguyên.
   Tính trên val/eval là rò rỉ thông tin của tập đánh giá vào tiền xử lý.
4. Toàn bộ tensor lên GPU một lần, tự xáo bằng `randperm` mỗi epoch (không `DataLoader`).

In [ ]:
if not os.path.exists(f"{REPO_ROOT}/data/processed/eval.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203

num = data["X_tr"][:, :10]
print("mean 10 cột số trên X_tr:", num.mean(0).cpu().numpy().round(4))
print("std  10 cột số trên X_tr:", num.std(0).cpu().numpy().round(4))
print("mean 10 cột số trên X_val (dùng thống kê của train, nên chỉ ≈ 0):", data["X_val"][:, :10].mean(0).cpu().numpy().round(3))

if SMOKE:  # kiểm thử nhanh: cắt nhỏ dữ liệu
    for k, n in (("X_tr", 20000), ("y_tr", 20000), ("X_val", 5000), ("y_val", 5000)):
        data[k] = data[k][:n]

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
`M-base` = `54 → 256 → 128 → 7`, ReLU, khởi tạo He (`kaiming_normal_`, bias = 0), **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# 1a/1b. Model, số tham số, shape qua từng lớp
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
assert n_params == EXPECTED_PARAMS[(256, 128)], n_params
print(model)
print("Số tham số:", n_params)

h = torch.randn(8, 54, device=device)
print("input", tuple(h.shape))
for layer in model.net:
    h = layer(h)
    print(f"  {layer.__class__.__name__:8s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# 1c-1. Loss bước 0 trên val (eval mode), kỳ vọng ≈ ln 7
step0 = evaluate(model, data["X_val"], data["y_val"])
print(f"loss bước 0 = {step0['loss']:.4f}   ln 7 = {LN7:.4f}   chênh = {step0['loss'] - LN7:+.4f}")
with torch.no_grad():
    p = torch.softmax(model(data["X_val"][:4096]), 1)
print("xác suất TB theo lớp ở bước 0:", p.mean(0).cpu().numpy().round(3), "(lý tưởng ≈ 1/7 = 0.143)")

In [ ]:
# 1c-2. Quá khớp 20 mẫu (dropout = 0): loss phải về ≈ 0
set_seed(0)
m20 = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
X20, y20 = data["X_tr"][:20], data["y_tr"][:20]
opt20 = torch.optim.Adam(m20.parameters(), lr=1e-3)
curve = []
for step in range(500):
    m20.train()
    loss = compute_loss(m20(X20), y20, "ce")
    opt20.zero_grad(); loss.backward(); opt20.step()
    curve.append(loss.item())
acc20 = (predict(m20, X20) == y20).float().mean().item()
print(f"loss bước 0 = {curve[0]:.4f}, sau 500 bước = {curve[-1]:.2e}, accuracy trên 20 mẫu = {acc20:.2f}")
print("các lớp trong 20 mẫu:", torch.bincount(y20, minlength=7).tolist())
plt.figure(figsize=(5, 3)); plt.semilogy(curve); plt.xlabel("bước"); plt.ylabel("CE loss (log)")
plt.title("Quá khớp 20 mẫu (Adam lr=1e-3)"); plt.grid(alpha=.3); plt.show()

In [ ]:
# 1d. Gradient có chảy tới mọi tham số?
set_seed(1)
mg = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
loss = compute_loss(mg(data["X_tr"][:512]), data["y_tr"][:512], "ce")
loss.backward()
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for nm, (pn, prm) in zip(names, mg.named_parameters()):
    assert prm.grad is not None and prm.grad.norm().item() > 0, pn
    print(f"{nm} ({pn:12s}) shape {tuple(prm.shape)!s:12s} ‖grad‖ = {prm.grad.norm().item():.4e}")
print("Mọi tham số có gradient khác 0.")

**Nhận xét Part 1:**
- Số tham số 47 879, logits `(8, 7)`: đúng quy định.
- Loss bước 0 = 2.269, cao hơn ln 7 = 1.946 khoảng 0.32. Nguyên nhân: He (`Var = 2/n_in`) được áp cả cho **lớp ra**, nơi không có ReLU phía sau,
  nên std của logit ≈ 0.58 (bảng ở Part 3.7) thay vì ≈ 0. Logit lệch ngẫu nhiên làm softmax không đều ⇒ CE > ln 7
  (xấp xỉ bậc hai: ln 7 + Var(z)·(1 − 1/7)/2 ≈ 1.946 + 0.14 ≈ 2.09; số đo 2.27 lớn hơn vì xấp xỉ bậc hai đánh giá thấp khi logit lệch xa 0). Mức lệch này vừa phải, không phải lỗi:
  `init-normal` (W ~ N(0, 0.01²)) cho đúng 1.946 và `init-zeros` cho đúng ln 7 = 1.9459. Nếu muốn khớp ln 7 với He, có thể thu nhỏ W lớp cuối (bảng "triệu chứng").
  Loss bước 0 cũng phụ thuộc seed (2.27 / 1.98 / 1.90 với seed 1/2/3, bảng `experiments.xlsx`).
- Quá khớp 20 mẫu: loss từ ≈ 1.97 về cỡ 1e-4 sau 500 bước Adam, accuracy 20/20 ⇒ pipeline forward/loss/backward/step đúng (nhãn, không softmax kép, có `zero_grad`).
- Mọi tham số W1, b1, W2, b2, W3, b3 có gradient khác 0 ⇒ gradient chảy tới mọi lớp.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` (trong `train.py`) là hàm duy nhất huấn luyện mọi thí nghiệm; mỗi thí nghiệm chỉ là một dict cfg.
Mỗi epoch ghi train loss (eval mode, 50 000 mẫu cố định của train), val loss/acc/macro-F1, `grad_norm` trước clip (TB và max),
thời gian (có `cuda.synchronize`), bộ nhớ cực đại; loss bước 0; best epoch theo val loss; cờ diverged.
`Runner.run` lưu `results/<exp_id>.json`, `figures/<exp_id>.png` và checkpoint của best epoch.

Baseline: `M-base`, He, CE, SGD + momentum 0.9, batch 512, 20 epoch, dropout 0, không clip, FP32.

**Chọn lr cho baseline bằng val:** thử lr ∈ {0.003, 0.01, 0.03, 0.1, 0.3} (seed 1), chọn lr có val macro-F1 cao nhất.
Các lần chạy này cũng là phần SGD+momentum của thí nghiệm bộ tối ưu (Part 3.2).

**Dự đoán:** lr 0.003 quá chậm (20 epoch chưa đủ), 0.3 có thể dao động hoặc phân kỳ với momentum 0.9 (lr hiệu dụng ≈ lr/(1−μ) = 3);
lr tốt nhất nằm khoảng 0.03–0.1.

In [ ]:
EPOCHS = 2 if SMOKE else 20
BASE = {**DEFAULT_CFG, "epochs": EPOCHS}
R = Runner(data, OUT_DIR, CKPT_DIR, verbose=False)

SGDM_LRS = [0.003, 0.01, 0.03, 0.1, 0.3]
sgdm = {}
for lr in SGDM_LRS:
    sgdm[lr] = R.run({**BASE, "exp_id": f"opt-sgdm-lr{lr:g}", "group": "optimizer",
                      "description": f"SGD+momentum 0.9, lr={lr:g} (dò lr cho baseline)", "lr": lr},
                     notes="Dò lr cho baseline; chọn theo val macro-F1.")
ok = {lr: r for lr, r in sgdm.items() if not r["summary"]["diverged"]}
BASE_LR = max(ok, key=lambda lr: ok[lr]["summary"]["val_macro_f1"])
print("\nlr baseline chọn bằng val:", BASE_LR)
R.table([f"opt-sgdm-lr{lr:g}" for lr in SGDM_LRS])

In [ ]:
# Baseline 3 seed. base-s1 trùng cấu hình với opt-sgdm-lr{BASE_LR} nên Runner dùng lại đúng lần chạy đó
# (không huấn luyện lại) và ta bỏ dòng opt-sgdm-lr{BASE_LR} khỏi bảng để mỗi lần chạy chỉ có một dòng.
BASE = {**BASE, "lr": BASE_LR}
SEEDS = [1, 2, 3]
for s in SEEDS:
    R.run({**BASE, "exp_id": f"base-s{s}", "group": "baseline", "seed": s,
           "description": f"Baseline M-base, SGD+momentum lr={BASE_LR:g}, seed {s}"},
          notes="Baseline; lr chọn bằng val từ dò lr SGD+momentum." + (" Cùng lần chạy với dò lr (seed 1)." if s == 1 else ""))
R.drop(f"opt-sgdm-lr{BASE_LR:g}")
sgdm[BASE_LR] = R.results["base-s1"]

base_ids = [f"base-s{s}" for s in SEEDS]
bt = R.table(base_ids)
display(bt)
NOISE = {m: (bt[m].mean(), bt[m].std(ddof=1)) for m in ("val_f1", "val_acc", "best_val_loss")}
for m, (mu, sd) in NOISE.items():
    print(f"{m:14s}: {mu:.4f} ± {sd:.4f}   (2σ = {2 * sd:.4f})")
F1_BASE, TWO_SIGMA = NOISE["val_f1"][0], 2 * NOISE["val_f1"][1]
plot_compare([R.results[i] for i in base_ids], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_baseline_seeds.png", "Baseline: 3 seed")
display(Image(f"{OUT_DIR}/figures/base-s1.png")); display(Image(f"{OUT_DIR}/figures/compare_baseline_seeds.png"))

In [ ]:
def vs_base(ids):
    # Bảng so sánh với baseline: delta val macro-F1 so với trung bình 3 seed baseline và có vượt 2σ không
    t = R.table(ids)
    t["Δf1_vs_base"] = (t["val_f1"] - F1_BASE).round(4)
    t["vượt_2σ"] = np.where(t["Δf1_vs_base"].abs() > TWO_SIGMA, "Có", "Không")
    return t

**Nhận xét baseline:**
- Dò lr cho SGD+momentum (seed 1, val macro-F1): 0.003 → 0.675, 0.01 → 0.765, 0.03 → 0.823, 0.1 → 0.850, **0.3 → 0.860**. Chọn **lr = 0.3** theo val.
  lr tốt nhất nằm ở **biên** lưới dò, nên lr tối ưu thật có thể lớn hơn một chút (hạn chế, nêu trong báo cáo).
- Baseline 3 seed: val macro-F1 = **0.8592 ± 0.0021** (2σ = **0.0042**), val acc = 0.9137 ± 0.0010, best val loss = 0.2176 ± 0.0027.
  Vượt xa mốc "đoán đa số" (acc 0.4876, macro-F1 ≈ 0.094).
- Hình dạng đường cong (`base-s1.png`): val loss giảm từ 0.443 (epoch 1) xuống 0.221 (epoch 20), có vài "răng cưa" (epoch 7, 10–11, 16, 19) do lr 0.3 khá lớn,
  best epoch = 20 (hoặc 19) ⇒ **chưa hội tụ**. Khoảng cách val − train loss chỉ 0.024 ⇒ **chưa quá khớp**, đang ở chế độ chưa khớp (underfit) theo thời gian huấn luyện.
  Hệ quả dự kiến: những gì tăng năng lực/số bước (mạng rộng, nhiều epoch) sẽ giúp; chính quy hoá (dropout) sẽ hại.
- grad_norm TB mỗi epoch ổn định quanh 0.34–0.40 (max theo bước 2.87 ở đầu huấn luyện).

## Part 3 — Thí nghiệm
Mỗi thí nghiệm đổi **một** yếu tố so với baseline (seed 1, 20 epoch, cùng phép tách val); nếu đổi hai yếu tố thì ghi rõ.
So sánh bằng val macro-F1 với ngưỡng nhiễu 2σ của 3 seed baseline.

### 3.1 Hàm mất mát — CE vs MSE (`loss-*`)
MSE tính trên **logit** (không softmax) với nhãn one-hot: `nn.MSELoss` mặc định, trung bình trên mọi phần tử (B × 7), không có hệ số 1/2.

**Dự đoán:** với CE, ∂L/∂z = (softmax(z) − y)/B: mẫu sai nặng vẫn có gradient lớn, không bão hoà.
Với MSE trên logit, ∂L/∂z = 2(z − y)/(7B): nhỏ hơn khoảng 3,5 lần ở cùng độ lệch, và MSE phạt cả logit "đúng quá mức".
Dự đoán MSE hội tụ chậm hơn và macro-F1 thấp hơn ở cùng lr; tăng lr ×3,5 (đổi 2 yếu tố, `loss-mse-lrx3.5`) bù được một phần.
Không so loss trực tiếp (khác thang đo), chỉ so acc/macro-F1.

In [ ]:
loss_ids = ["loss-mse", "loss-mse-lrx3.5"]
R.run({**BASE, "exp_id": "loss-mse", "group": "loss", "loss": "mse",
       "description": "MSE trên logit với one-hot (thay CE)"},
      notes="Dự đoán: hội tụ chậm hơn CE vì gradient MSE nhỏ hơn ~3.5x và không ưu tiên mẫu sai nặng.")
R.run({**BASE, "exp_id": "loss-mse-lrx3.5", "group": "loss", "loss": "mse", "lr": BASE_LR * 3.5,
       "description": f"MSE + lr x3.5 (= {BASE_LR * 3.5:g}); ĐỔI 2 YẾU TỐ: bù độ lớn gradient"},
      notes="Đổi 2 yếu tố (loss, lr) có chủ đích: kiểm tra chênh lệch có chỉ do độ lớn gradient không.")
plot_compare([R.results["base-s1"]] + [R.results[i] for i in loss_ids], ["val_acc", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_loss.png", "CE vs MSE (không so loss vì khác thang đo)")
display(vs_base(["base-s1"] + loss_ids)); display(Image(f"{OUT_DIR}/figures/compare_loss.png"))

**Đối chiếu (loss):**
- `loss-mse` (cùng lr 0.3): val macro-F1 **0.7794**, thấp hơn baseline 0.080 (≫ 2σ = 0.0042), accuracy 0.881 vs 0.913. Khớp dự đoán: gradient của MSE trên logit
  nhỏ (grad_norm TB 0.068 so với 0.353 của CE, ~5 lần) và không "tăng tốc" cho mẫu sai nặng ⇒ học chậm, các lớp hiếm bị bỏ qua nhiều hơn (macro-F1 giảm mạnh hơn accuracy).
- `loss-mse-lrx3.5` (lr 1.05, đổi 2 yếu tố): **khác dự đoán** — mạng sụp về dự đoán hằng số: acc 0.4876, macro-F1 0.0936 (đoán toàn lớp 1).
  Có một gai gradient 14.2 ở epoch 1, sau đó grad_norm ≈ 0.04 và val loss đứng yên ở **0.0889**, đúng bằng MSE tối thiểu của một dự đoán hằng = tần suất lớp:
  (1 − Σp_c²)/7 = (1 − 0.378)/7 ≈ 0.089. Tức là bước quá lớn đẩy các nơ-ron ReLU vào vùng âm (ReLU chết), chỉ còn bias lớp cuối học tần suất lớp.
  Kết luận: tăng lr không bù được cho MSE; vấn đề của MSE không chỉ là độ lớn gradient.

### 3.2 Bộ tối ưu hoá (`opt-*`)
Mỗi bộ thử 3 lr cách nhau ~3 lần, so ở **lr tốt nhất theo val** của từng bộ. SGD+momentum dùng lại 5 lần dò lr ở Part 2.
- SGD (không momentum): lr ∈ {0.03, 0.1, 0.3}
- Adam (β=(0.9, 0.999), ε=1e-8): lr ∈ {3e-4, 1e-3, 3e-3}
- AdamW (như Adam, weight_decay = 0.01 tách riêng): lr ∈ {3e-4, 1e-3, 3e-3}
- Kiểm chứng: AdamW với weight_decay = 0 phải trùng Adam (cùng lr 1e-3, cùng seed).

**Dự đoán:** SGD thường cần lr lớn hơn SGD+momentum khoảng 1/(1−μ) = 10 lần và vẫn chậm hơn trong 20 epoch.
Adam xuống nhanh ở vài epoch đầu nhờ chia bước theo √v̂ từng tham số; ở lr tốt nhất mỗi bộ, chênh lệch Adam vs SGD+momentum có thể nhỏ (gần mức nhiễu).
AdamW wd = 0.01 gần như không khác Adam vì mô hình chưa quá khớp.

In [ ]:
OPT_GRID = {"sgd": [0.03, 0.1, 0.3], "adam": [3e-4, 1e-3, 3e-3], "adamw": [3e-4, 1e-3, 3e-3]}
opt_runs = {"sgd_momentum": sgdm}
for name, lrs in OPT_GRID.items():
    opt_runs[name] = {}
    for lr in lrs:
        wd = 0.01 if name == "adamw" else 0.0
        opt_runs[name][lr] = R.run({**BASE, "exp_id": f"opt-{name}-lr{lr:g}", "group": "optimizer",
                                    "optimizer": name, "lr": lr, "weight_decay": wd,
                                    "description": f"{name}, lr={lr:g}, wd={wd:g}"},
                                   notes="So sánh bộ tối ưu: mỗi bộ ≥ 3 lr, so ở lr tốt nhất theo val.")
R.run({**BASE, "exp_id": "opt-adamw-wd0-lr0.001", "group": "optimizer", "optimizer": "adamw", "lr": 1e-3,
       "weight_decay": 0.0, "description": "AdamW wd=0, lr=1e-3 (kiểm chứng trùng Adam)"},
      notes="Kiểm chứng: AdamW với wd=0 phải cho cùng kết quả với Adam lr=1e-3.")

best_lr = {}
for name, runs in opt_runs.items():
    ok = {lr: r for lr, r in runs.items() if not r["summary"]["diverged"]}
    best_lr[name] = max(ok, key=lambda lr: ok[lr]["summary"]["val_macro_f1"])
best_ids = {n: runs[best_lr[n]]["cfg"]["exp_id"] for n, runs in opt_runs.items()}
print("lr tốt nhất theo val:", best_lr)
display(vs_base(list(best_ids.values())))
BEST_OPT = max(best_ids, key=lambda n: R.results[best_ids[n]]["summary"]["val_macro_f1"])
print("Bộ tối ưu tốt nhất theo val:", BEST_OPT, "lr =", best_lr[BEST_OPT])

a, b = R.results["opt-adam-lr0.001"]["history"], R.results["opt-adamw-wd0-lr0.001"]["history"]
print("max |Δ val_loss| Adam vs AdamW(wd=0):", float(np.max(np.abs(np.array(a["val_loss"]) - np.array(b["val_loss"])))))

plot_compare([R.results[i] for i in best_ids.values()], ["val_loss", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_optimizer.png", "Mỗi bộ tối ưu ở lr tốt nhất (theo val)")
for name, runs in opt_runs.items():
    plot_compare([runs[lr] for lr in sorted(runs)], ["val_loss", "val_macro_f1"],
                 f"{OUT_DIR}/figures/compare_optimizer_lr_{name}.png", f"{name}: độ nhạy theo lr")
display(Image(f"{OUT_DIR}/figures/compare_optimizer.png"))
opt_tbl = vs_base([r["cfg"]["exp_id"] for runs in opt_runs.values() for r in runs.values()] + ["opt-adamw-wd0-lr0.001"])
display(opt_tbl)

**Đối chiếu (optimizer):** (lr tốt nhất theo val của mỗi bộ, seed 1; 2σ = 0.0042)

| bộ tối ưu | lr thử | lr tốt nhất | val macro-F1 | Δ vs TB baseline |
|---|---|---|---|---|
| SGD | 0.03 / 0.1 / 0.3 | 0.3 | 0.8016 | −0.058 |
| SGD+momentum 0.9 | 0.003 … 0.3 | 0.3 (`base-s1`) | 0.8603 | +0.001 |
| Adam | 3e-4 / 1e-3 / 3e-3 | 3e-3 | **0.8762** | **+0.017** |
| AdamW (wd 0.01) | 3e-4 / 1e-3 / 3e-3 | 3e-3 | 0.8697 | +0.010 |

- Ở lr tốt nhất của mỗi bộ: **Adam > AdamW > SGD+momentum ≫ SGD**; Adam hơn SGD+momentum 0.016, gấp ~4 lần 2σ ⇒ khác biệt có ý nghĩa (với giả định nhiễu seed của Adam tương tự baseline).
  Khác dự đoán ở chỗ khoảng cách lớn hơn mức "gần nhiễu" mình đoán.
- Cơ chế: Adam chia bước theo √v̂ của từng tham số nên các trọng số nối với cột one-hot hiếm (Soil_Type ít xuất hiện, gradient nhỏ) vẫn được cập nhật đủ lớn ⇒ có lợi cho lớp hiếm và macro-F1.
- Nếu không chỉnh lr công bằng thì kết luận đảo ngược: Adam ở 3e-4 (0.796) thua SGD+momentum ở 0.3 (0.860), và SGD+momentum ở 0.003 (0.675) thua cả SGD thường ở 0.3.
- SGD thường cần lr lớn hơn ~10 lần: `opt-sgd-lr0.3` (0.802) gần `opt-sgdm-lr0.03` (0.823), khớp lr hiệu dụng của momentum ≈ lr/(1 − μ). SGD lr 0.3 dao động mạnh
  (best epoch 16, macro-F1 epoch cuối chỉ 0.716).
- `opt-adamw-wd0-lr0.001` trùng **từng chữ số** với `opt-adam-lr0.001` (max |Δ val_loss| = 0) ⇒ xác nhận Adam ≡ AdamW khi wd = 0.
- AdamW wd 0.01 kém Adam 0.0065 ở lr 3e-3 (vượt 2σ một chút): suy giảm tách riêng thu nhỏ W mỗi bước một hệ số (1 − lr·wd); qua 14 540 bước ≈ e^(−0.44) ≈ 0.65 — đáng kể với một mô hình đang **chưa khớp**, nên chính quy hoá chỉ có hại.
- Hạn chế: lr tốt nhất của SGD, SGD+momentum và Adam đều ở biên trên của lưới dò; mỗi lr chỉ 1 seed.

### 3.3 Hyper-parameter (`hp-*`)
- Batch 128 và 2048 (cùng lr, cùng 20 epoch ⇒ số bước cập nhật ×4 và ÷4: 2 906 và 182 bước/epoch so với 727).
- Batch 2048 + lr ×4 (quy tắc tăng lr theo lô; đổi 2 yếu tố có chủ đích, không khởi động).
- Độ rộng `M-wide` (54→512→256→7, 161 287 tham số) và độ sâu `M-deep` (54→256→128→64→7, 55 687 tham số).
- 40 epoch (cùng baseline) để xem 20 epoch đã hội tụ chưa.

**Dự đoán:** batch 128 tốt hơn ở cùng số epoch (nhiều bước hơn) nhưng chậm hơn mỗi epoch; batch 2048 cùng lr tệ hơn rõ vì chỉ có 1/4 số bước;
tăng lr ×4 bù lại phần lớn. `M-wide` tốt hơn (đang chưa khớp, thêm năng lực có ích), `M-deep` hơn baseline ít. 40 epoch tốt hơn 20 vì đường val còn giảm.

In [ ]:
hp_cfgs = [
    ("hp-batch128", dict(batch=128), "batch 128 (cùng lr)", ""),
    ("hp-batch2048", dict(batch=2048), "batch 2048 (cùng lr)", ""),
    ("hp-batch2048-lrx4", dict(batch=2048, lr=BASE_LR * 4), f"batch 2048 + lr x4 = {BASE_LR * 4:g}; ĐỔI 2 YẾU TỐ (linear scaling)", "Đổi 2 yếu tố có chủ đích (batch, lr)."),
    ("hp-wide", dict(hidden=(512, 256)), "M-wide 512-256", ""),
    ("hp-deep", dict(hidden=(256, 128, 64)), "M-deep 256-128-64", ""),
    ("hp-epochs40", dict(epochs=2 * EPOCHS), "40 epoch (baseline 20)", "Khác số epoch có chủ đích: kiểm tra đã hội tụ chưa."),
]
for eid, ch, desc, note in hp_cfgs:
    R.run({**BASE, "exp_id": eid, "group": "hparam", "description": desc, **ch}, notes=note)
hp_ids = [c[0] for c in hp_cfgs]
t = vs_base(["base-s1"] + hp_ids)
t["steps/epoch"] = [R.results[i]["summary"]["steps_per_epoch"] for i in t.index]
display(t)
plot_compare([R.results[i] for i in ["base-s1", "hp-batch128", "hp-batch2048", "hp-batch2048-lrx4"]],
             ["val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_hparam_batch.png", "Batch size (cùng số epoch ≠ cùng số bước)")
plot_compare([R.results[i] for i in ["base-s1", "hp-wide", "hp-deep", "hp-epochs40"]],
             ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_hparam_arch.png", "Độ rộng / độ sâu / số epoch")
display(Image(f"{OUT_DIR}/figures/compare_hparam_batch.png")); display(Image(f"{OUT_DIR}/figures/compare_hparam_arch.png"))

**Đối chiếu (hparam):** (Δ so với TB baseline 0.8592; 2σ = 0.0042)
- `hp-batch2048` (cùng lr): 0.8381 (−0.021). Khớp dự đoán: chỉ 182 bước/epoch (so với 727) ⇒ 1/4 số bước cập nhật trong 20 epoch.
- `hp-batch2048-lrx4` (lr 1.2): 0.7599 (−0.099). **Khác dự đoán**: quy tắc tăng lr theo lô thất bại vì lr gốc 0.3 đã gần biên ổn định; lr 1.2 không khởi động (warmup)
  tạo gai gradient 14.8 ở epoch 1, val loss epoch 1 = 0.996 rồi hồi phục chậm. Slide nêu quy tắc này đi kèm khởi động, mình đã bỏ qua phần đó.
- `hp-batch128` (cùng lr): 0.7851 (−0.074). **Khác dự đoán** (đoán tốt hơn vì nhiều bước hơn): lô nhỏ ⇒ gradient nhiễu hơn ~2 lần (độ lệch chuẩn ∝ 1/√B), với lr 0.3 + momentum
  bước cập nhật quá ồn: val loss dao động, best epoch 12 rồi xấu dần. Lô nhỏ cần lr nhỏ hơn tương ứng. Thời gian/epoch tăng 3.6 → 7.2 s (2 906 bước/epoch).
- `hp-wide` (512-256): **0.8765 (+0.017)**, vượt nhiễu. Khớp dự đoán: mô hình đang chưa khớp, tăng năng lực (161k vs 48k tham số) giảm cả train loss (0.171 vs 0.196) lẫn val loss.
  Đổi lại thời gian/epoch 9.5 s vs 3.6 s.
- `hp-deep` (256-128-64): 0.8504 (−0.009), vượt 2σ theo chiều xấu. Khác dự đoán: thêm một lớp hẹp 64 chỉ thêm 7.8k tham số nhưng làm đường val dao động hơn ở lr 0.3
  (best epoch 17, epoch cuối 0.842); lr chọn cho M-base có thể không tối ưu cho M-deep.
- `hp-epochs40`: **0.8798 (+0.021)**, best epoch = 40. Khớp dự đoán: 20 epoch chưa hội tụ; vẫn chưa quá khớp ở 40 epoch (gap val − train 0.035).

### 3.4 Dropout (`drop-*`)
q ∈ {0.1, 0.3, 0.5} trên `M-base`, đặt sau ReLU của mỗi lớp ẩn. Train loss đo ở `eval()` nên so được với val loss.

**Dự đoán:** baseline 20 epoch gần như chưa quá khớp (dự kiến khoảng cách val − train loss rất nhỏ), nên dropout chủ yếu làm giảm năng lực hiệu dụng
và làm chậm học: val macro-F1 giảm dần khi q tăng, đặc biệt ở các lớp hiếm. Dropout là thuốc cho quá khớp, không phải cho chưa khớp.

In [ ]:
drop_ids = []
for q in (0.1, 0.3, 0.5):
    eid = f"drop-{q:g}"
    R.run({**BASE, "exp_id": eid, "group": "dropout", "dropout": q, "description": f"dropout q={q:g} sau ReLU các lớp ẩn"},
          notes="Dự đoán: baseline chưa quá khớp nên dropout làm giảm val macro-F1.")
    drop_ids.append(eid)
t = vs_base(["base-s1"] + drop_ids)
t["gap_val_train"] = (t["val_loss"] - t["train_loss"]).round(4)
display(t)
plot_compare([R.results[i] for i in ["base-s1"] + drop_ids], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_dropout.png", "Dropout (train loss đo ở eval mode)")
display(Image(f"{OUT_DIR}/figures/compare_dropout.png"))

**Đối chiếu (dropout):** khớp dự đoán.

| exp | q | val macro-F1 | Δ | final train loss | final val loss | gap val − train |
|---|---|---|---|---|---|---|
| base-s1 | 0 | 0.8603 | +0.001 | 0.196 | 0.221 | 0.024 |
| drop-0.1 | 0.1 | 0.8388 | −0.021 | 0.228 | 0.242 | 0.014 |
| drop-0.3 | 0.3 | 0.7711 | −0.088 | 0.316 | 0.320 | 0.004 |
| drop-0.5 | 0.5 | 0.6526 | −0.207 | 0.418 | 0.421 | 0.003 |

- Khoảng cách val − train thu hẹp khi q tăng (dropout đúng là chống quá khớp), nhưng **cả hai** loss đều tăng ⇒ mô hình vốn chưa quá khớp nên dropout chỉ làm giảm năng lực hiệu dụng
  và làm gradient nhiễu hơn. Mọi mức q đều kém baseline vượt 2σ; lớp hiếm chịu thiệt nhất (macro-F1 giảm nhanh hơn accuracy: 0.913 → 0.825 khi q = 0.5).
- Dropout nên dùng khi train loss thấp mà val loss tăng dần (quá khớp), ví dụ mạng rộng huấn luyện rất lâu. Ở `final-c` (M-wide, 40 epoch) q = 0.1 vẫn kém không dropout (0.9189 vs 0.9242).

### 3.5 Gradient clipping (`clip-*`)
`c` chọn từ `grad_norm` của baseline: đặt bằng **trung vị của grad_norm TB mỗi epoch** của `base-s1`, để clip thực sự kích hoạt ở một phần đáng kể các bước
(cột `clip_frac` trong history ghi tỉ lệ bước bị cắt).
Thí nghiệm phản chứng: lr ×10 so với baseline, chạy **không clip** và **có clip c**.

**Dự đoán:** ở lr baseline, clipping chỉ làm bước nhỏ đi một chút ⇒ khác biệt trong mức nhiễu. Ở lr ×10, không clip sẽ dao động mạnh hoặc NaN
(gai grad_norm lớn), có clip thì bước cập nhật bị chặn ⇒ huấn luyện ổn định hơn, val macro-F1 cao hơn rõ.

In [ ]:
gn = np.array(R.results["base-s1"]["history"]["grad_norm"])
gn_max = np.array(R.results["base-s1"]["history"]["grad_norm_max"])
CLIP_C = float(f"{np.median(gn):.2g}")
print("base-s1 grad_norm TB/epoch: min %.3f, median %.3f, max %.3f; max theo bước: %.3f" % (gn.min(), np.median(gn), gn.max(), gn_max.max()))
print("c chọn =", CLIP_C)
HI_LR = BASE_LR * 10
clip_cfgs = [
    (f"clip-c{CLIP_C:g}", dict(clip_norm=CLIP_C), f"clip c={CLIP_C:g} ở lr baseline"),
    (f"clip-none-lr{HI_LR:g}", dict(lr=HI_LR), f"KHÔNG clip, lr x10 = {HI_LR:g} (đối chứng)"),
    (f"clip-c{CLIP_C:g}-lr{HI_LR:g}", dict(lr=HI_LR, clip_norm=CLIP_C), f"clip c={CLIP_C:g}, lr x10 = {HI_LR:g}"),
]
for eid, ch, desc in clip_cfgs:
    R.run({**BASE, "exp_id": eid, "group": "clipping", "description": desc, **ch},
          notes=f"c = trung vị grad_norm TB/epoch của base-s1. " + ("Cặp lr cao: chỉ khác nhau ở clip." if "lr" in ch else ""))
clip_ids = [c[0] for c in clip_cfgs]
t = vs_base(["base-s1"] + clip_ids)
t["clip_frac_TB"] = [np.mean(R.results[i]["history"]["clip_frac"]) for i in t.index]
display(t)
plot_compare([R.results[i] for i in ["base-s1"] + clip_ids], ["val_loss", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_clipping.png", f"Clipping (c={CLIP_C:g}); grad_norm đo trước clip")
display(Image(f"{OUT_DIR}/figures/compare_clipping.png"))

**Đối chiếu (clipping):**
- Chọn c từ grad_norm của `base-s1`: TB mỗi epoch nằm trong 0.339–0.395 (trung vị 0.349) ⇒ **c = 0.35**. Ở c này clipping **thực sự kích hoạt ở 68% số bước**.
- `clip-c0.35` (lr 0.3): 0.8564, Δ = −0.003, **nằm trong 2σ** ⇒ không kết luận được khác biệt. Khớp dự đoán: khi gradient không có gai, clip chỉ thu nhỏ bước một chút.
- Thí nghiệm phản chứng ở lr = 3.0 (×10):
  - `clip-none-lr3`: gai grad_norm **163.5** ở epoch 1, sau đó mạng sụp về dự đoán toàn lớp 1 (macro-F1 0.0936; val loss ≈ 1.207 = entropy của phân phối lớp
    −Σ p ln p ≈ 1.205 ⇒ chỉ còn bias lớp cuối hoạt động, các ReLU đã chết). Không ra NaN nên cờ diverged = N, nhưng thực chất là hỏng.
  - `clip-c0.35-lr3`: gai bị chặn (max 4.7 trước clip), mạng **không** chết hẳn nhưng vẫn rất tệ (macro-F1 tốt nhất 0.20 ở epoch 1, sau đó dao động).
  - **Khác dự đoán**: clipping không cứu được lr quá lớn. Lý do: clip chặn độ dài ‖g‖ ≤ c nhưng bước cập nhật vẫn là lr·c = 1.05 mỗi bước, và momentum 0.9 cộng dồn tới ~10 lần.
    Clipping chữa các gai **hiếm** (rare spikes), không chữa được một lr sai một bậc độ lớn. Thử nghiệm tốt hơn: lr ×2–3 (gần biên ổn định) thay vì ×10.

### 3.6 Mixed precision (`amp-*`)
- FP16: `autocast(float16)` + `GradScaler` (unscale trước khi đo/clip gradient). **Chạy trên GPU**, so với mốc FP32 cũng chạy trên GPU (`amp-fp32-gpu`, `amp-wide-fp32-gpu`),
  vì các thí nghiệm khác chạy trên CPU (thời gian không so được).
- BF16: `autocast(bfloat16)`, không GradScaler. T4 không có phần cứng BF16 (compute capability 7.5) nên có thể chậm hoặc lỗi; nếu lỗi sẽ ghi lý do.
- Thêm `M-wide` FP16 so với `hp-wide` (FP32) để xem mạng lớn hơn có lợi không.
Tham số vẫn ở FP32, loss tính ở FP32 (`compute_loss` ép `.float()`).

**Dự đoán:** độ chính xác như FP32 (trong mức nhiễu). Thời gian/epoch **không** giảm đáng kể với `M-base` vì mạng nhỏ (ma trận 512×256),
thời gian bị chi phối bởi chi phí khởi chạy kernel và các thao tác autocast; có thể còn chậm hơn. BF16 trên T4 chậm hơn FP32 vì phải giả lập.

In [ ]:
# AMP phải đo trên GPU. Các thí nghiệm khác đã chạy trên CPU, nên để so thời gian/bộ nhớ công bằng,
# chạy lại FP32 trên GPU làm mốc (amp-fp32-gpu, amp-wide-fp32-gpu). Cờ runtime="cuda" tách cache khỏi các lần chạy CPU.
import traceback
AMP = {**BASE, "group": "amp", "runtime": "cuda"}
amp_cfgs = [("amp-fp32-gpu", dict(), "FP32 trên GPU: mốc so sánh cho FP16/BF16 (cùng cấu hình baseline)"),
            ("amp-fp16", dict(precision="fp16"), "FP16 autocast + GradScaler"),
            ("amp-bf16", dict(precision="bf16"), "BF16 autocast (không GradScaler)"),
            ("amp-wide-fp32-gpu", dict(hidden=(512, 256)), "M-wide FP32 trên GPU: mốc cho amp-wide-fp16; ĐỔI 2 YẾU TỐ so với baseline (kiến trúc)"),
            ("amp-wide-fp16", dict(precision="fp16", hidden=(512, 256)), "M-wide + FP16; ĐỔI 2 YẾU TỐ so với baseline (kiến trúc, precision)")]
amp_ids = []
if device != "cuda":
    print("Bỏ qua AMP: cần GPU CUDA (Runtime → Change runtime type → T4 GPU).")
else:
    for eid, ch, desc in amp_cfgs:
        try:
            R.run({**AMP, "exp_id": eid, "description": desc, **ch}, require_device="cuda",
                  notes="Chạy trên GPU " + torch.cuda.get_device_name(0) + "; so thời gian/epoch, bộ nhớ, độ chính xác với mốc FP32 trên GPU.")
            amp_ids.append(eid)
        except Exception:
            print(f"{eid} LỖI:"); traceback.print_exc()
if amp_ids:
    t = vs_base(amp_ids)
    display(t[["prec", "hidden", "best_val_loss", "val_acc", "val_f1", "Δf1_vs_base", "vượt_2σ", "s_per_ep", "mem_MB"]])
    plot_compare([R.results[i] for i in amp_ids], ["val_loss", "val_macro_f1"],
                 f"{OUT_DIR}/figures/compare_amp.png", "FP32 vs FP16 vs BF16 (tất cả trên GPU)")
    display(Image(f"{OUT_DIR}/figures/compare_amp.png"))

**Đối chiếu (amp):** _(điền sau khi chạy)_

### 3.7 Khởi tạo tham số (`init-*`)
`zeros` (W = 0), `normal` (N(0, 0.01²)), `xavier` (`xavier_normal_`, Var = 2/(n_in+n_out)), `default` (mặc định của `nn.Linear`:
kaiming_uniform(a=√5), Var = 1/(3·n_in)), baseline `he` (Var = 2/n_in). Bias = 0 (trừ `default`).

**Dự đoán:**
- `zeros`: mọi tiền kích hoạt = 0 ⇒ ReLU(0) = 0 ⇒ gradient của W1, W2 bằng 0 (ReLU'(0) = 0 trong PyTorch), chỉ bias lớp cuối học được ⇒
  mạng chỉ học được phân phối tiên nghiệm của lớp: accuracy ≈ 0.4876, macro-F1 ≈ 0.09. Loss bước 0 đúng bằng ln 7.
- `normal` 0.01: kích hoạt co lại ~×(0.01·√n_in·√(1/2)) mỗi lớp ⇒ logit ≈ 0, loss bước 0 ≈ ln 7, gradient nhỏ ⇒ học chậm ở đầu nhưng mạng 3 lớp vẫn thoát được.
- `xavier`/`default`: std kích hoạt giảm dần qua lớp (thiếu hệ số 2 cho ReLU), với 3 lớp chênh lệch nhỏ ⇒ kết quả gần He (trong mức nhiễu).

In [ ]:
# Độ lệch chuẩn kích hoạt (sau mỗi Linear) và loss bước 0 cho từng cách khởi tạo, M-base, một lô val 4096 mẫu
xb = data["X_val"][:4096]
rows = []
for init in ("zeros", "normal", "xavier", "default", "he"):
    set_seed(1)
    m = MLP(hidden=(256, 128), init=init).to(device)
    stds = activation_stats(m, xb)
    rows.append(dict(init=init, std_L1=stds[0], std_L2=stds[1], std_logit=stds[2],
                     step0_loss=evaluate(m, data["X_val"], data["y_val"])["loss"]))
display(pd.DataFrame(rows).set_index("init").round(4))

# Mạng sâu 30 lớp ẩn 256 (chỉ để quan sát kích hoạt như slide, KHÔNG huấn luyện, không nằm trong bảng)
plt.figure(figsize=(7, 4))
for init in ("he", "xavier", "default", "normal"):
    set_seed(1)
    deep = MLP(hidden=(256,) * 30, init=init).to(device)
    plt.semilogy(range(1, 32), activation_stats(deep, xb), "o-", ms=3, label=init)
plt.xlabel("lớp Linear thứ"); plt.ylabel("std tiền kích hoạt (log)"); plt.grid(alpha=.3); plt.legend()
plt.title("MLP 30 lớp ẩn ReLU, bước 0: std kích hoạt theo độ sâu")
plt.savefig(f"{OUT_DIR}/figures/compare_init_actstd.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
init_ids = []
for init in ("zeros", "normal", "xavier", "default"):
    eid = f"init-{init}"
    R.run({**BASE, "exp_id": eid, "group": "init", "init": init, "description": f"khởi tạo {init} (baseline: he)"},
          notes="zeros: dự đoán không học được do đối xứng + ReLU(0)." if init == "zeros" else "")
    init_ids.append(eid)
display(vs_base(["base-s1"] + init_ids))
plot_compare([R.results[i] for i in ["base-s1"] + init_ids], ["val_loss", "val_macro_f1", "grad_norm"],
             f"{OUT_DIR}/figures/compare_init.png", "Khởi tạo tham số")
display(Image(f"{OUT_DIR}/figures/compare_init.png"))

**Đối chiếu (init):**
- Bước 0 (`M-base`, lô val 4 096 mẫu, std sau mỗi Linear):

| init | std L1 | std L2 | std logit | loss bước 0 | val macro-F1 (20 ep) |
|---|---|---|---|---|---|
| zeros | 0 | 0 | 0 | 1.9459 | 0.0936 |
| normal 0.01 | 0.034 | 0.0038 | 0.0003 | 1.9460 | 0.8602 |
| xavier | 0.276 | 0.218 | 0.192 | 2.0222 | 0.8604 |
| default | 0.274 | 0.113 | 0.059 | 1.9830 | 0.8603 |
| **he** (baseline) | 0.661 | 0.640 | 0.577 | 2.2691 | 0.8603 |

- `zeros`: **khớp dự đoán**. Mọi kích hoạt = 0 ⇒ ReLU(0) = 0 và ReLU'(0) = 0 ⇒ gradient của W1, W2, W3 đều bằng 0, chỉ b3 học được ⇒ mạng chỉ học tần suất lớp:
  acc 0.4876, macro-F1 0.0936, val loss 1.205 (= entropy phân phối lớp), grad_norm TB 0.038. Dù có lấy được gradient, mọi nơ-ron cùng lớp vẫn nhận cùng cập nhật (đối xứng) nên không bao giờ khác nhau.
- `normal` / `xavier` / `default`: **cùng kết quả với He trong mức nhiễu** (Δ ≤ 0.0012 < 2σ = 0.0042). Với 3 lớp, std co lại theo lớp (normal: 0.034 → 0.0003) nhưng
  chưa tới mức triệt tiêu, và SGD+momentum lr 0.3 nhanh chóng đưa trọng số về thang đo phù hợp. Khác biệt duy nhất thấy được: grad_norm cực đại ở đầu (normal 0.78, default 0.90, xavier 1.24, he 2.87).
- Mạng 30 lớp ẩn (ảnh `compare_init_actstd.png`, chỉ bước 0): std ở lớp 30 — **He ≈ 0.42** (giữ thang đo), **Xavier ≈ 7.6e-6** (giảm ×√2 mỗi lớp vì
  Var = 2/(n_in+n_out) = 1/n thiếu hệ số 2 bù cho ReLU), **normal 0.01 ≈ 8e-30** (triệt tiêu hoàn toàn), `default` ≈ 0.038 (W co lại nhưng bias khởi tạo đều
  U(±1/√n) giữ kích hoạt ở thang của bias). Đây là lý do He quan trọng với mạng sâu dùng ReLU; mạng 3 lớp của lab quá nông để thấy khác biệt khi huấn luyện.

### Kết hợp — cấu hình cuối (`final-*`), chọn **chỉ bằng val**
Kết hợp những gì vượt nhiễu trên val ở Part 3: bộ tối ưu tốt nhất (ở lr tốt nhất của nó), 40 epoch, cosine annealing theo bước (lr giảm dần về 0 giúp
hội tụ cuối), và thử hai kiến trúc + dropout nhẹ cho mạng rộng (mạng rộng + nhiều epoch có thể bắt đầu quá khớp):
- `final-a`: `M-base` + bộ tối ưu tốt nhất + 40 epoch + cosine
- `final-b`: `M-wide` + bộ tối ưu tốt nhất + 40 epoch + cosine
- `final-c`: như `final-b` + dropout 0.1

Chọn ứng viên có val macro-F1 cao nhất, chạy thêm seed 2 và 3 để đo nhiễu. File nộp dùng seed 1.

In [ ]:
FIN = {**BASE, "optimizer": BEST_OPT, "lr": best_lr[BEST_OPT], "weight_decay": 0.01 if BEST_OPT == "adamw" else 0.0,
       "epochs": 2 * EPOCHS, "scheduler": "cosine", "group": "final"}
fin_cands = {"final-a": dict(hidden=(256, 128)), "final-b": dict(hidden=(512, 256)),
             "final-c": dict(hidden=(512, 256), dropout=0.1)}
for k, ch in fin_cands.items():
    R.run({**FIN, **ch, "exp_id": f"{k}-s1", "description": f"Ứng viên cuối {k}: {BEST_OPT} lr={FIN['lr']:g}, 40 ep, cosine, {ch}"},
          notes="Ứng viên cấu hình cuối; ĐỔI NHIỀU YẾU TỐ so với baseline (kết hợp). Chọn bằng val macro-F1.")
FINAL_KEY = max(fin_cands, key=lambda k: R.results[f"{k}-s1"]["summary"]["val_macro_f1"])
print("Cấu hình cuối chọn bằng val:", FINAL_KEY)
for s in (2, 3):
    R.run({**FIN, **fin_cands[FINAL_KEY], "seed": s, "exp_id": f"{FINAL_KEY}-s{s}",
           "description": f"Cấu hình cuối {FINAL_KEY}, seed {s}"}, notes="Cấu hình cuối, seed khác để đo nhiễu.")
final_ids = [f"{FINAL_KEY}-s{s}" for s in (1, 2, 3)]
ft = vs_base([f"{k}-s1" for k in fin_cands] + final_ids[1:])
display(ft)
fs = R.table(final_ids)["val_f1"]
print(f"Cấu hình cuối val macro-F1: {fs.mean():.4f} ± {fs.std(ddof=1):.4f}  vs baseline {F1_BASE:.4f} ± {NOISE['val_f1'][1]:.4f}")
plot_compare([R.results[i] for i in ["base-s1"] + [f"{k}-s1" for k in fin_cands]], ["train_loss", "val_loss", "val_macro_f1"],
             f"{OUT_DIR}/figures/compare_final.png", "Ứng viên cấu hình cuối (chọn bằng val)")
display(Image(f"{OUT_DIR}/figures/compare_final.png"))

## Part 4 — Đánh giá cuối trên eval, bảng, nộp bài
Chỉ chạy **sau** khi đã chọn cấu hình bằng val. Nạp trọng số epoch có val loss thấp nhất, dự đoán toàn bộ eval ở `eval()`,
chấm bằng đúng `scripts/evaluate.py`. Không chỉnh cấu hình sau khi thấy điểm eval.
- `predictions_eval.csv` + `eval_result.json` (nộp): cấu hình cuối, seed 1.
- `eval_result_baseline.json` (bổ sung): baseline `base-s1`; file dự đoán baseline để ở `checkpoints/` (không nộp).

In [ ]:
FINAL_ID = f"{FINAL_KEY}-s1"
out_abs = os.path.abspath(OUT_DIR)
ck_abs = os.path.abspath(CKPT_DIR)

def run_eval(exp_id, pred_path, json_path):
    r = R.results[exp_id]
    final_eval(r["cfg"], r, data, pred_path)
    p = subprocess.run([sys.executable, "scripts/evaluate.py", "--pred", pred_path, "--out", json_path],
                       cwd=REPO_ROOT, capture_output=True, text=True)
    print(f"===== {exp_id} =====\n" + p.stdout + p.stderr)
    with open(json_path) as f:
        return json.load(f)

if SMOKE:
    print("SMOKE: bỏ qua eval đầy đủ"); EVAL = {}
else:
    EVAL = {"base-s1": run_eval("base-s1", f"{ck_abs}/predictions_eval_baseline.csv", f"{out_abs}/eval_result_baseline.json"),
            FINAL_ID: run_eval(FINAL_ID, f"{out_abs}/predictions_eval.csv", f"{out_abs}/eval_result.json")}
    for k, v in EVAL.items():
        print(f"{k:14s} eval acc = {v['accuracy']:.4f}  eval macro-F1 = {v['macro_f1']:.4f}  "
              f"(val macro-F1 = {R.results[k]['summary']['val_macro_f1']:.4f})")

In [ ]:
# Phân tích lỗi theo lớp (cấu hình cuối, từ eval_result.json) + so sánh với baseline
NAMES = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
if EVAL:
    ev, evb = EVAL[FINAL_ID], EVAL["base-s1"]
    pc = pd.DataFrame(ev["per_class"]).set_index("cls")
    pc.insert(0, "name", NAMES)
    pc["f1_baseline"] = [c["f1"] for c in evb["per_class"]]
    display(pc.round(4))
    cm = np.array(ev["confusion_matrix"])
    cmn = cm / cm.sum(1, keepdims=True)
    off = cmn.copy(); np.fill_diagonal(off, 0)
    for c in np.argsort(pc["f1"].values)[:3]:
        j = off[c].argmax()
        print(f"Lớp {c} ({NAMES[c]}): F1={pc.loc[c, 'f1']:.4f}, support={pc.loc[c, 'support']}, "
              f"nhầm nhiều nhất sang lớp {j} ({NAMES[j]}): {off[c, j]:.1%} số mẫu thật của lớp {c}")
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set(xticks=range(7), yticks=range(7), xlabel="dự đoán", ylabel="nhãn thật",
           title=f"Ma trận nhầm lẫn trên eval ({FINAL_ID}), chuẩn hoá theo hàng")
    fig.colorbar(im); fig.savefig(f"{OUT_DIR}/figures/compare_eval_confusion.png", dpi=110, bbox_inches="tight"); plt.show()

**Kết quả chọn bằng val:** `final-a` 0.9104, **`final-b` 0.9242**, `final-c` 0.9189 ⇒ chọn `final-b` (Adam lr 3e-3, M-wide 512-256, 40 epoch, cosine, không dropout).
3 seed: val macro-F1 **0.9248 ± 0.0009** so với baseline 0.8592 ± 0.0021 (+0.066, ≫ 2σ).

**Đánh giá cuối (eval, `scripts/evaluate.py`):**

| cấu hình | val macro-F1 | eval macro-F1 | eval acc |
|---|---|---|---|
| baseline `base-s1` | 0.8603 | 0.8666 | 0.9126 |
| cuối `final-b-s1` (file nộp) | 0.9242 | **0.9280** | **0.9530** |

Cải thiện trên eval +0.061 macro-F1. Val và eval rất gần (chênh 0.004–0.006) ⇒ val là ước lượng tốt của eval.

**Phân tích lỗi (cấu hình cuối, eval):**
- Lớp khó nhất: **lớp 4 Aspen (F1 0.878**, support 1 899) và **lớp 3 Cottonwood/Willow (F1 0.880**, support 549 — lớp hiếm nhất, 0.47%).
- Aspen bị nhầm nhiều nhất sang **lớp 1 Lodgepole Pine (197 mẫu = 10.4%)**. Trên tập train: độ cao Aspen 2 788 ± 97 m nằm gọn trong dải của Lodgepole 2 921 ± 186 m,
  và cả hai đều nằm ở Wilderness_Area 0 và 2 ⇒ đặc trưng chồng lấn; Lodgepole là lớp đa số (48.8%, nhiều hơn Aspen ~30 lần) nên vùng chồng lấn bị kéo về phía nó.
- Cottonwood/Willow bị nhầm sang **lớp 2 Ponderosa Pine (48 mẫu = 8.7%)** và lớp 5 Douglas-fir (27). Trên train: 100% Cottonwood ở Wilderness_Area 3, nơi cũng chứa 60% Ponderosa
  và 56% Douglas-fir; độ cao 2 224 ± 102 m gần Ponderosa (2 394 ± 196) và Douglas-fir (2 419 ± 189). Lớp 3 chỉ có ~1 760 mẫu trong phần train sau khi tách val.
- Theo số tuyệt đối, cặp nhầm lớn nhất là **lớp 0 Spruce/Fir ↔ lớp 1 Lodgepole Pine** (2 082 và 1 746 mẫu): hai lớp lớn nhất, độ cao chồng lấn (3 129 ± 158 vs 2 921 ± 186 m), cùng các vùng hoang dã 0 và 2.
- So với baseline, mọi lớp đều cải thiện, nhiều nhất ở lớp hiếm: Aspen 0.769 → 0.878, Douglas-fir 0.818 → 0.915, Cottonwood 0.823 → 0.880.
- Cách cải thiện có thể thử: trọng số lớp trong CE hoặc lấy mẫu cân bằng cho lớp 3/4, huấn luyện lâu hơn / mạng lớn hơn (đường val của `final-b` vẫn giảm tới epoch 39–40).

In [ ]:
# Bảng experiments.xlsx từ results/*.json (eval chỉ điền cho baseline và cấu hình cuối)
results = load_results(f"{OUT_DIR}/results")
order = ["baseline", "loss", "optimizer", "hparam", "dropout", "clipping", "amp", "init", "final"]
results.sort(key=lambda r: (order.index(r["cfg"]["group"]), r["cfg"]["exp_id"]))
rows = []
for r in results:
    row = to_row(r, EVAL.get(r["cfg"]["exp_id"]))
    if r["summary"].get("device") != "cuda":   # kết quả cũ không có khoá device: đều chạy trên CPU
        row["notes"] = (row["notes"] + " Chạy trên CPU (Colab không bật GPU): time_per_epoch_s là thời gian CPU, peak_mem_MB để trống vì không có bộ nhớ GPU.").strip()
    rows.append(row)
SUMMARY_NOTES = {
    "baseline": "SGD+momentum lr 0.3 (chọn bằng val). 3 seed: val macro-F1 0.8592 ± 0.0021 (2σ 0.0042). Chưa hội tụ (best epoch 19–20), chưa quá khớp.",
    "loss": "MSE trên logit kém CE 0.08 macro-F1 ở cùng lr; MSE lr x3.5 sụp về dự đoán hằng (ReLU chết).",
    "optimizer": "Ở lr tốt nhất mỗi bộ: Adam 3e-3 (0.876) > AdamW (0.870) > SGD+mom 0.3 (0.860) >> SGD 0.3 (0.802). AdamW wd=0 trùng Adam.",
    "hparam": "M-wide +0.017 và 40 epoch +0.021 (vượt nhiễu). Batch 128/2048 và linear scaling lr x4 (không warmup) đều kém. M-deep −0.009.",
    "dropout": "Mọi q làm giảm macro-F1 (−0.02 đến −0.21): gap val−train thu hẹp nhưng cả hai loss tăng ⇒ mô hình chưa quá khớp.",
    "clipping": "c=0.35 (trung vị grad_norm) cắt 68% bước, khác biệt trong nhiễu. Ở lr x10 clip chặn gai 163 nhưng không cứu được huấn luyện.",
    "amp": "Chạy trên GPU, so với mốc FP32 trên GPU (xem notebook/báo cáo).",
    "init": "zeros không học (đối xứng + ReLU(0)); normal/xavier/default ≈ He trong nhiễu với mạng 3 lớp; khác biệt rõ ở mạng 30 lớp (bước 0).",
    "final": "Chọn bằng val: final-b (Adam 3e-3, M-wide, 40 ep, cosine) val 0.9248 ± 0.0009; eval macro-F1 0.9280 (baseline 0.8666).",
}
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=base_ids, summary_notes=SUMMARY_NOTES)
figs = {f[:-4] for f in os.listdir(f"{OUT_DIR}/figures") if f.endswith(".png") and not f.startswith("compare_")}
ids = {r["exp_id"] for r in rows}
print(f"{len(rows)} dòng trong bảng, {len(figs)} ảnh <exp_id>.png; khớp: {figs == ids}")
assert figs == ids, (figs ^ ids)
pd.DataFrame(rows).set_index("exp_id")[["group", "optimizer", "lr", "batch", "hidden", "dropout", "clip_norm", "precision",
                                        "init", "best_epoch", "val_acc", "val_macro_f1", "eval_acc", "eval_macro_f1"]]